<a href="https://colab.research.google.com/github/barry-clarke/CS5004/blob/main/Etivity3_24325082_Tasks3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install gymnasium[atari] ale-py

In [5]:
import gymnasium as gym
import ale_py
import numpy as np
import tensorflow as tf
from collections import deque

# 1. Environment Setup (Atari Breakout)
# Explicitly register the Atari environments to fix the 'Namespace ALE' error
gym.register_envs(ale_py)

env = gym.make("ALE/Breakout-v5")
n_outputs = env.action_space.n

# 2. Convolutional Neural Network (CNN) for Pixel Inputs
# Architecture mirrors the DeepMind Nature (2015) paper
# Input shape assumes preprocessed, stacked frames: (84, 84, 4)
input_shape = [84, 84, 4]

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=input_shape),
    # 3 Convolutional layers to extract spatial features from the pixels
    tf.keras.layers.Conv2D(32, kernel_size=8, strides=4, activation="relu"),
    tf.keras.layers.Conv2D(64, kernel_size=4, strides=2, activation="relu"),
    tf.keras.layers.Conv2D(64, kernel_size=3, strides=1, activation="relu"),
    tf.keras.layers.Flatten(),
    # Fully connected layers to map features to Q-values
    tf.keras.layers.Dense(512, activation="relu"),
    tf.keras.layers.Dense(n_outputs)
])

# 3. Optimiser and Loss Function
optimizer = tf.keras.optimizers.Adam(learning_rate=0.00025, clipnorm=1.0)
loss_fn = tf.keras.losses.Huber() # Huber loss is standard for Atari stability

# 4. The Core Training Step
@tf.function
def training_step(states, actions, rewards, next_states, dones):
    # Ask the model for the next states' Q-values
    next_Q_values = model(next_states, training=False)
    max_next_Q_values = tf.reduce_max(next_Q_values, axis=1)

    runs = 1.0 - tf.cast(dones, tf.float32)
    rewards = tf.cast(rewards, tf.float32)

    # ----------------------------------------------------------------
    # 1. WHERE THE TARGET IS CALCULATED
    # The target is calculated here using the Bellman equation.
    # It combines the immediate reward with the discounted maximum
    # expected future reward (gamma = 0.99).
    # ----------------------------------------------------------------

    # --- TARGET CALCULATION LOCATED HERE ---
    target_Q_values = rewards + runs * 0.99 * max_next_Q_values
    target_Q_values = tf.reshape(target_Q_values, [-1, 1])
    # ---------------------------------------

    mask = tf.one_hot(actions, n_outputs)

    with tf.GradientTape() as tape:
        all_Q_values = model(states, training=True)
        Q_values = tf.reduce_sum(all_Q_values * mask, axis=1, keepdims=True)
        loss = tf.reduce_mean(loss_fn(target_Q_values, Q_values))

    # --- WEIGHT UPDATE LOCATED HERE ---
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    # ----------------------------------

In [6]:
# --- 5. Output for Verification ---
print("--- ENVIRONMENT DETAILS ---")
print(f"Environment: {env.spec.id}")
print(f"Observation Space (Raw): {env.observation_space}")
print(f"Action Space: {env.action_space} (Total Actions: {n_outputs})")

print("\n--- CNN ARCHITECTURE SUMMARY ---")
# This prints the layer-by-layer breakdown of the CNN
model.summary()

print("\n--- FORWARD PASS TEST ---")
# Pass a dummy state (1 batch, 84x84 pixels, 4 stacked frames) through the network
dummy_state = np.zeros((1, 84, 84, 4), dtype=np.float32)
dummy_q_values = model(dummy_state)
print(f"Output Q-Values shape: {dummy_q_values.shape}")
print(f"Output Q-Values: {dummy_q_values.numpy()}")

--- ENVIRONMENT DETAILS ---
Environment: ALE/Breakout-v5
Observation Space (Raw): Box(0, 255, (210, 160, 3), uint8)
Action Space: Discrete(4) (Total Actions: 4)

--- CNN ARCHITECTURE SUMMARY ---


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 20, 20, 32)     │         8,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 9, 9, 64)       │        32,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 7, 7, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │     1,606,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │         2,052 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,686,180 (6.43 MB)

 Trainable params: 1,686,180 (6.43 MB)

 Non-trainable params: 0 (0.00 B)


--- FORWARD PASS TEST ---
Output Q-Values shape: (1, 4)
Output Q-Values: [[0. 0. 0. 0.]]


### Analysis of the Atari DQN Architecture and Environment

**1. Environment Specifications**
The output confirms `ALE/Breakout-v5` has loaded successfully. The raw environment provides a standard colour image (210x160 pixels). The raw images are too heavy to process directly so are, convert to grayscale, shrinking to 84x84, and stacking 4 frames together so the AI can "see" motion. The output also confirms the action space is `Discrete(4)`, which perfectly matches the 4 possible joystick movements in Breakout.

**2. CNN Architecture Summary**
The model summary proves we built the Convolutional Neural Network (CNN) exactly as described in the DeepMind (2015) paper.
* **Conv2D Layers:** The three convolutional layers act as the "eyes", scanning the raw pixels to identify important game features (like the ball and paddle).
* **Dense Layer:** This visual data is then flattened and fed into a massive 512-neuron "brain".

This network is vastly more complex than the Acrobot model, with over 1.6 million trainable parameters.

**3. Forward Pass Verification**
To prove the computational graph works before starting a massive training loop, a blank "dummy" image was passed through the network. The output shape `(1, 4)` mathematically proves the network successfully takes one state and predicts exactly four Q-values (one for each action). The output of `[[0. 0. 0. 0.]]` is perfectly normal—it simply confirms the network is randomly initialised and hasn't started learning yet!